<a href="https://colab.research.google.com/github/dinujakr/Final-year-project-IEKF-for-rigid-body-tracking/blob/main/so3_ekf_lib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
so3_ekf_lib.py
==============
Library containing:
  - LinearKF          : Joseph-stabilized linear Kalman filter
  - SO3IMUSensorFusionBiasEKF : Intrinsic error-state EKF on SO(3) with gyro bias

Process model (error-state on SO(3)):
    Error state:  eta = [eta_R ; eta_b] in R^6
    Nominal:      R_k^- = R_{k-1}^+ exp(dt * hat(Omega_k - b_k^-))
                  b_k^- = b_{k-1}^+
    Measurement:  y_k = [R_k^T e_1 ; R_k^T e_2 ; ...]
"""

import numpy as np


# ================================================================
#  Custom exceptions
# ================================================================
class KFShapeError(ValueError):
    pass


class KFValueError(ValueError):
    pass


# ================================================================
#  LinearKF
# ================================================================
class LinearKF:
    """Linear-Gaussian Kalman Filter with explicit predict/update.

    Process model:  x_k = A_k x_{k-1} + G_k w_{k-1},   w ~ N(0, Q)
    Measurement:    y_k = H_k x_k + z_k,                z ~ N(0, R)
    """

    def __init__(self, *, use_joseph: bool = True, symmetrize: bool = True,
                 atol_sym: float = 1e-10):
        self.use_joseph = use_joseph
        self.symmetrize = symmetrize
        self.atol_sym = atol_sym

    # ---------- public API ----------

    def predict(self, m_prev, P_prev, A, Q, G=None):
        """One-step prediction."""
        m_prev = self._as_1d(m_prev, "m_prev")
        P_prev = self._as_2d(P_prev, "P_prev")
        A = self._as_2d(A, "A")
        Q = self._as_2d(Q, "Q")
        n_prev = m_prev.shape[0]

        if P_prev.shape != (n_prev, n_prev):
            raise KFShapeError(
                f"`P_prev` must be {(n_prev, n_prev)}; got {P_prev.shape}.")
        if A.shape[1] != n_prev:
            raise KFShapeError(
                f"`A` must have second dim {n_prev}; got {A.shape}.")

        n = A.shape[0]
        self._assert_symmetric(P_prev, "P_prev", atol=self.atol_sym)
        self._assert_symmetric(Q, "Q", atol=self.atol_sym)
        _ = self._assert_spd(Q, "Q")

        if G is None:
            if Q.shape != (n, n):
                raise KFShapeError(
                    f"With G=None, `Q` must be shape {(n, n)}; got {Q.shape}.")
            m_pred = A @ m_prev
            P_pred = A @ P_prev @ A.T + Q
        else:
            G = self._as_2d(G, "G")
            if G.shape[0] != n:
                raise KFShapeError(
                    f"`G` must have {n} rows; got {G.shape}.")
            r = G.shape[1]
            if Q.shape != (r, r):
                raise KFShapeError(
                    f"With G shape {(n, r)}, `Q` must be {(r, r)}; got {Q.shape}.")
            m_pred = A @ m_prev
            P_pred = A @ P_prev @ A.T + G @ Q @ G.T

        if self.symmetrize:
            P_pred = 0.5 * (P_pred + P_pred.T)

        return m_pred, P_pred

    def measurement_update(self, m_pred, P_pred, H, y, R):
        """Measurement update → (m_upd, P_upd, K, S)."""
        m_pred, P_pred, H, y, R, n, p = self._validate_shapes(
            m_pred, P_pred, H, y, R)

        self._assert_symmetric(P_pred, "P_pred", atol=self.atol_sym)
        self._assert_symmetric(R, "R", atol=self.atol_sym)
        _ = self._assert_spd(R, "R")

        HP = H @ P_pred
        S = HP @ H.T + R
        try:
            Ls = np.linalg.cholesky(S)
        except np.linalg.LinAlgError as e:
            raise KFValueError(
                f"Innovation covariance `S` is not SPD. "
                f"Check H, P_pred, and R. Cholesky failed: {e}")

        U = np.linalg.solve(Ls, HP)
        X = np.linalg.solve(Ls.T, U)
        K = X.T

        v = y - H @ m_pred
        m_upd = m_pred + K @ v

        I = np.eye(n)
        if self.use_joseph:
            I_KH = I - K @ H
            P_upd = I_KH @ P_pred @ I_KH.T + K @ R @ K.T
        else:
            P_upd = (I - K @ H) @ P_pred

        if self.symmetrize:
            P_upd = 0.5 * (P_upd + P_upd.T)

        return m_upd, P_upd, K, S

    def step(self, m_prev, P_prev, A, Q, H, y, R, G=None):
        """Convenience: predict → update in one call."""
        m_pred, P_pred = self.predict(m_prev, P_prev, A, Q, G)
        m_upd, P_upd, K, S = self.measurement_update(
            m_pred, P_pred, H, y, R)
        return m_upd, P_upd, K, S, (m_pred, P_pred)

    # ---------- helpers ----------

    @staticmethod
    def _as_1d(x, name):
        x = np.asarray(x, dtype=float)
        if x.ndim == 2 and x.shape[1] == 1:
            x = x[:, 0]
        if x.ndim != 1:
            raise KFShapeError(
                f"`{name}` must be 1D; got shape {x.shape}.")
        return x

    @staticmethod
    def _as_2d(x, name):
        x = np.asarray(x, dtype=float)
        if x.ndim != 2:
            raise KFShapeError(
                f"`{name}` must be 2D; got {x.ndim}D with shape {x.shape}.")
        return x

    @staticmethod
    def _assert_symmetric(M, name, atol=1e-10):
        if M.shape[0] != M.shape[1]:
            raise KFShapeError(f"`{name}` must be square; got {M.shape}.")
        if not np.allclose(M, M.T, atol=atol):
            raise KFValueError(
                f"`{name}` must be symmetric within atol={atol}.")

    @staticmethod
    def _assert_spd(M, name):
        try:
            return np.linalg.cholesky(M)
        except np.linalg.LinAlgError as e:
            raise KFValueError(
                f"`{name}` must be SPD. Cholesky failed: {e}")

    def _validate_shapes(self, m_pred, P_pred, H, y, R):
        m_pred = self._as_1d(m_pred, "m_pred")
        y = self._as_1d(y, "y")
        P_pred = self._as_2d(P_pred, "P_pred")
        H = self._as_2d(H, "H")
        R = self._as_2d(R, "R")

        n = m_pred.shape[0]
        if P_pred.shape != (n, n):
            raise KFShapeError(
                f"`P_pred` shape mismatch: expected {(n, n)}, got {P_pred.shape}.")
        if H.shape[1] != n:
            raise KFShapeError(
                f"`H` second dim must be {n}; got {H.shape}.")
        p = H.shape[0]
        if y.shape[0] != p:
            raise KFShapeError(
                f"`y` length must be {p}; got {y.shape}.")
        if R.shape != (p, p):
            raise KFShapeError(
                f"`R` must be {(p, p)}; got {R.shape}.")
        return m_pred, P_pred, H, y, R, n, p


# ================================================================
#  SO3IMUSensorFusionBiasEKF
# ================================================================
class SO3IMUSensorFusionBiasEKF:
    """Error-state intrinsic EKF on SO(3) with gyro bias estimation.

    Error state:
        eta_k = [eta_R,k ; eta_b,k] in R^6

    Nominal propagation:
        R_k^- = R_{k-1}^+ exp(dt * hat(Omega_k - b_k^-))
        b_k^- = b_{k-1}^+

    Measurement:
        y_k = [R_k^T e_1; R_k^T e_2; ...]
    """

    def __init__(self, kf, inertial_directions, dt):
        self.kf = kf
        self.E = np.asarray(inertial_directions, dtype=float)  # (m, 3)
        self.dt = float(dt)

        if self.E.ndim != 2 or self.E.shape[1] != 3:
            raise ValueError("`inertial_directions` must have shape (m,3).")

    # ---- SO(3) helpers ----

    @staticmethod
    def hat(x):
        """Skew-symmetric (hat) map: R^3 → so(3)."""
        x = np.asarray(x, dtype=float).reshape(3)
        return np.array([
            [0.0, -x[2], x[1]],
            [x[2], 0.0, -x[0]],
            [-x[1], x[0], 0.0],
        ])

    @classmethod
    def exp_so3(cls, omega):
        """Exponential map: so(3) → SO(3) via Rodrigues."""
        omega = np.asarray(omega, dtype=float).reshape(3)
        theta = np.linalg.norm(omega)
        W = cls.hat(omega)

        if theta < 1e-12:
            return np.eye(3) + W + 0.5 * W @ W

        A = np.sin(theta) / theta
        B = (1.0 - np.cos(theta)) / theta ** 2
        return np.eye(3) + A * W + B * W @ W

    @classmethod
    def Phi_so3(cls, omega):
        """Left Jacobian approximation (truncated Baker-Campbell-Hausdorff)."""
        omega = np.asarray(omega, dtype=float).reshape(3)
        W = cls.hat(omega)
        return np.eye(3) + 0.5 * W + (1.0 / 12.0) * W @ W

    def predict_measurement(self, R_minus):
        """Predicted stacked direction measurements: yhat = [R^T e_1; ...]."""
        R_minus = np.asarray(R_minus, dtype=float).reshape(3, 3)
        return np.concatenate([R_minus.T @ e for e in self.E])

    # ---- Linearisation matrices ----

    def define_AGH(self, R_minus, Omega_k, b_minus, R_prev_plus=None):
        """
        Build time-varying A, G, H for the error-state [eta_R; eta_b].

        Returns
        -------
        A : (6,6)
        G : (6,6)
        H : (3m, 6)
        """
        R_minus = np.asarray(R_minus, dtype=float).reshape(3, 3)
        Omega_k = np.asarray(Omega_k, dtype=float).reshape(3)
        b_minus = np.asarray(b_minus, dtype=float).reshape(3)

        R_for_G = R_minus if R_prev_plus is None else np.asarray(
            R_prev_plus, dtype=float).reshape(3, 3)

        Omega_corr = Omega_k - b_minus

        # A: identity coupling with bias
        A = np.block([
            [np.eye(3), -self.dt * np.eye(3)],
            [np.zeros((3, 3)), np.eye(3)],
        ])

        # G: noise input matrix
        G_R = -self.dt * R_for_G @ self.Phi_so3(-self.dt * Omega_corr)
        G = np.block([
            [G_R, np.zeros((3, 3))],
            [np.zeros((3, 3)), np.eye(3)],
        ])

        # H: measurement Jacobian
        H_blocks = []
        for e in self.E:
            yhat_i = R_minus.T @ e
            H_blocks.append(
                np.hstack([self.hat(yhat_i), np.zeros((3, 3))])
            )
        H = np.vstack(H_blocks)

        return A, G, H

    # ---- Main filter loop ----

    def run_filter(self, R0_plus, P0, Omegas, Ys, Sigma_q, Sigma_m,
                   b0_plus=None, store_corrected_yhat=True):
        """
        Run the SO(3) gyro-bias error-state EKF.

        Parameters
        ----------
        R0_plus   : (3,3)   Initial corrected attitude.
        P0        : (6,6)   Initial error covariance for [eta_R; eta_b].
        Omegas    : (T,3)   Gyroscope measurements.
        Ys        : (T,3m)  Stacked vector measurements.
        Sigma_q   : (6,6)   Process noise cov [gyro; bias random walk].
        Sigma_m   : (3m,3m) Measurement noise covariance.
        b0_plus   : (3,)    Initial gyro bias estimate.
        store_corrected_yhat : bool   Store yhat after or before correction.

        Returns
        -------
        results : dict
        """
        R_plus = np.asarray(R0_plus, dtype=float).reshape(3, 3)
        b_plus = (np.zeros(3) if b0_plus is None
                  else np.asarray(b0_plus, dtype=float).reshape(3))
        P = np.asarray(P0, dtype=float).reshape(6, 6)

        Omegas = np.asarray(Omegas, dtype=float)
        Ys = np.asarray(Ys, dtype=float)

        T = Omegas.shape[0]
        if Ys.shape[0] != T:
            raise ValueError(
                "`Omegas` and `Ys` must have the same number of steps.")

        m_dirs = self.E.shape[0]
        if Ys.shape[1] != 3 * m_dirs:
            raise ValueError(
                f"`Ys` must have shape (T,{3 * m_dirs}); got {Ys.shape}.")
        if Sigma_m.shape != (3 * m_dirs, 3 * m_dirs):
            raise ValueError(
                f"`Sigma_m` must be {(3 * m_dirs, 3 * m_dirs)}; "
                f"got {Sigma_m.shape}.")
        if Sigma_q.shape != (6, 6):
            raise ValueError(
                f"`Sigma_q` must be (6,6); got {Sigma_q.shape}.")

        # Storage
        R_plus_list = []
        R_minus_list = []
        b_plus_list = []
        b_minus_list = []
        yhat_list = []
        K_list = []
        P_list = []
        S_list = []

        m_err = np.zeros(6)

        for k in range(T):
            Omega_k = Omegas[k]
            y_k = Ys[k]

            # Nominal prediction
            b_minus = b_plus.copy()
            Omega_corr = Omega_k - b_minus
            R_minus = R_plus @ self.exp_so3(self.dt * Omega_corr)

            # Time-varying linearisation
            A, G, H = self.define_AGH(
                R_minus=R_minus,
                Omega_k=Omega_k,
                b_minus=b_minus,
                R_prev_plus=R_plus,
            )

            # Predicted measurement and residual
            yhat_minus_k = self.predict_measurement(R_minus)
            residual_k = y_k - yhat_minus_k

            # Linear KF update on local error state
            m_upd, P_upd, K, S, (m_pred, P_pred) = self.kf.step(
                m_prev=m_err,
                P_prev=P,
                A=A,
                Q=Sigma_q,
                G=G,
                H=H,
                y=residual_k,
                R=Sigma_m,
            )

            # Split correction
            delta_R = m_upd[:3]
            delta_b = m_upd[3:]

            # Inject correction into SO(3) and bias
            R_plus = R_minus @ self.exp_so3(delta_R)
            b_plus = b_minus + delta_b

            # Store predicted or corrected yhat
            if store_corrected_yhat:
                yhat_k = self.predict_measurement(R_plus)
            else:
                yhat_k = yhat_minus_k

            # Reset local error mean after injection
            m_err = np.zeros(6)
            P = P_upd

            R_minus_list.append(R_minus)
            R_plus_list.append(R_plus)
            b_minus_list.append(b_minus)
            b_plus_list.append(b_plus)
            yhat_list.append(yhat_k)
            K_list.append(K)
            P_list.append(P)
            S_list.append(S)

        return {
            "R_minus": np.asarray(R_minus_list),
            "R_plus": np.asarray(R_plus_list),
            "b_minus": np.asarray(b_minus_list),
            "b_plus": np.asarray(b_plus_list),
            "yhat": np.asarray(yhat_list),
            "K": K_list,
            "P": P_list,
            "S": S_list,
        }

    # ---- Plotting ----

    def plot_measurements(self, Ys, Yhat, title="Measured vs predicted directions"):
        """Plot measured y_k and predicted yhat_k using matplotlib."""
        import matplotlib.pyplot as plt

        Ys = np.asarray(Ys, dtype=float)
        Yhat = np.asarray(Yhat, dtype=float)

        if Ys.shape != Yhat.shape:
            raise ValueError(
                f"`Ys` and `Yhat` must have same shape; "
                f"got {Ys.shape} and {Yhat.shape}.")

        T, d = Ys.shape
        t = np.arange(T)

        fig, axes = plt.subplots(d, 1, figsize=(12, 2.5 * d), sharex=True)
        if d == 1:
            axes = [axes]

        for j in range(d):
            axes[j].plot(t, Ys[:, j], linewidth=0.8, label=f"y[{j}] measured")
            axes[j].plot(t, Yhat[:, j], linewidth=0.8, linestyle="--",
                         label=f"y[{j}] predicted")
            axes[j].set_ylabel(f"Component {j}")
            axes[j].legend(loc="upper right", fontsize=8)
            axes[j].grid(True, alpha=0.3)

        axes[-1].set_xlabel("Time step k")
        fig.suptitle(title, fontsize=13, fontweight="bold")
        plt.tight_layout()
        return fig
